In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import base64

load_dotenv()


True

## Encode the image and send to the vision model

In [2]:
with open("blood_report.png","rb") as f:
    image_64 = base64.b64encode(f.read()).decode()
image_64[:200]

'iVBORw0KGgoAAAANSUhEUgAAAWUAAAG1CAIAAACu2XLDAAAAA3NCSVQICAjb4U/gAAAa/3pUWHRSYXcgcHJvZmlsZSB0eXBlIEFQUDEAAGiBtZpZklw5ckX/axW9BMDhmJaDUdZmMknW+//QuYggi6wu05+YVczMYMR7gOP6HYD3x3+c/zr/+uf6x//867/vP//z/PEP'

### The above converting image to string

In [3]:
llm = ChatGroq(
    model="qwen/qwen3.8-27b",
)

In [5]:
message = HumanMessage(content=[
    {"type":"image_url","image_url":{"url":f"data:image/png;base64,{image_64}"}},
    {"type":"text","text":"This is a blood work report. Extract all the test results and flag any values outside the normal range"}
])
response = llm.invoke([message])
print(response.content)

Based on the blood work report provided, here is the extraction of all test results and any values outside the normal range:

### Patient Information
- **Patient:** Rajesh Sharma
- **Age:** 48
- **Gender:** Male
- **Date:** May 7, 2026

### Complete Blood Count (CBC)
- **Hemoglobin:** 15.1 g/dL (Normal: 13.5-17.5) - Within normal range
- **Hematocrit:** 44% (Normal: 41-53%) - Within normal range
- **WBC:** 6.8 x10^3/uL (Normal: 4.5-11.0) - Within normal range
- **Platelets:** 220 x10^3/uL (Normal: 150-400) - Within normal range

### Lipid Panel
- **Total Cholesterol:** 238 mg/dL (Normal: <200) - **Outside normal range**
- **LDL Cholesterol:** 162 mg/dL (Normal: <100) - **Outside normal range**
- **HDL Cholesterol:** 36 mg/dL (Normal: >40) - **Outside normal range**
- **Triglycerides:** 188 mg/dL (Normal: <150) - **Outside normal range**

### Metabolic Panel
- **Glucose (Fasting):** 92 mg/dL (Normal: 70-99) - Within normal range
- **HbA1c:** 5.3% (Normal: <5.7%) - Within normal range
- 

In [6]:
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import create_react_agent


@tool
def get_diet_recommendation(abnormal_findings: str) -> str:
  """Provide tailored dietary recommendations based on abnormal blood work findings."""
  # Add your logic or database queries here
  return (
      f"Dietary recommendations based on findings ({abnormal_findings}): Increase"
      " intake of iron-rich foods, maintain proper hydration, and reduce "
      "processed sugars."
  )

tools = [get_diet_recommendation]

# 5. Create the multi-modal agent using LangGraph's prebuilt agent
checkpointer = InMemorySaver()
agent = create_react_agent(
    model=llm,
    tools=tools,
    checkpointer=checkpointer,
    prompt=(
        "You are a helpful medical and nutrition assistant. Analyze the blood"
        " work and use the available tools when dietary advice is requested."
    ),
)

# 6. Invoke the agent incorporating the instructor's suggestion (using the previous response)
agent_input = {
    "messages": [
        message,
        response,
        HumanMessage(
            content=(
                "Based on these results, please call the get_diet_recommendation"
                " tool to provide tailored advice."
            )
        ),
    ]
}

config = {"configurable": {"thread_id": "1"}}

# Run the agent and stream the response
for step in agent.stream(agent_input, config, stream_mode="updates"):
  for node_name, node_output in step.items():
    print(f"--- Step: {node_name} ---")
    print(node_output)


C:\Users\DELL\AppData\Local\Temp\ipykernel_22292\1955639440.py:20: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


--- Step: agent ---
{'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'tgcb0800w', 'function': {'arguments': '{"abnormal_findings":"Total Cholesterol: 238 mg/dL (High), LDL Cholesterol: 162 mg/dL (High), HDL Cholesterol: 36 mg/dL (Low), Triglycerides: 188 mg/dL (High)"}', 'name': 'get_diet_recommendation'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 2241, 'total_tokens': 2328, 'completion_time': 0.329561672, 'completion_tokens_details': None, 'prompt_time': 0.164796476, 'prompt_tokens_details': None, 'queue_time': 0.372504964, 'total_time': 0.494358148}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1825d-2d53-757a-b6a3-f53e58d2cc5c-0', tool_calls=[{'name': 'get_diet_recommendation', 'args': {'abnormal_findings': 'Total Cholesterol: 238 mg/dL (High), LDL Chol